<a href="https://colab.research.google.com/github/miguel-rs7/Rag-radiografias/blob/main/github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!git clone https://github.com/miguel-rs7/rag-radiografias.git
%cd rag-radiografias
!pip install -q -r requirements.txt
from google.colab import drive
drive.mount('/content/drive')

Cloning into 'rag-radiografias'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 49 (delta 16), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 352.16 KiB | 5.97 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/rag-radiografias
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which i

In [5]:
import sys; sys.path.insert(0, "/content/rag-radiografias/repo")
import kagglehub
from google.colab import userdata
from src import embeddings
from src.db import conectar
from src.rag import muestra, indexar, buscar, evaluar

embeddings.iniciar(userdata.get("GEMINI_API_KEY"))
col, db = conectar()
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.


In [6]:
rutas = muestra(path, "train", 100)
indexar(col, db, rutas, "train")
for id_, diag, dist in buscar(col, db, rutas[0]):
    print(id_, diag, dist)
print(f"Acierto: {evaluar(col, db, muestra(path, 'test', 25)):.0%}")

200 ya indexadas, 0 pendientes
NORMAL_IM-0115-0001 NORMAL 0.0
NORMAL_IM-0227-0001 NORMAL 0.006
NORMAL_IM-0278-0001 NORMAL 0.006
NORMAL_IM-0269-0001 NORMAL 0.006
NORMAL_IM-0135-0001 NORMAL 0.006
Acierto: 90%


In [8]:
from google import genai
from google.colab import userdata

cliente = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

print("Conexión con Gemini configurada.")

respuesta = cliente.models.generate_content(
    model="gemini-3.8-flash",
    contents="Explica brevemente qué significa NORMAL y PNEUMONIA en un conjunto de datos de radiografías de tórax. Es una explicación educativa, no un diagnóstico médico."
)

print(respuesta.text)

Conexión con Gemini configurada.
En el contexto de la ciencia de datos y el aprendizaje automático, estas etiquetas se utilizan para clasificar las imágenes en dos categorías (generalmente para tareas de clasificación binaria):

### 1. **NORMAL (Sano / Control)**
* **Significado:** Representa a pacientes cuyos pulmones no presentan signos visuales de infección activa ni anomalías graves en la imagen.
* **Aspecto visual:** Los pulmones se ven predominantemente **oscuros o negros**. Esto se debe a que están llenos de aire, el cual no bloquea los rayos X. Las estructuras normales (como las costillas, la silueta del corazón y los vasos sanguíneos) se observan con bordes nítidos y definidos.

### 2. **PNEUMONIA (Neumonía / Patológico)**
* **Significado:** Representa a pacientes con una infección en uno o ambos pulmones (comúnmente causada por bacterias o virus) que inflama los sacos aéreos (alvéolos).
* **Aspecto visual:** Se observan **manchas blancas, sombras difusas o zonas "nubladas"** 

In [10]:

# Tomar una radiografía de ejemplo
consulta = rutas[0]

# Recuperar las radiografías más similares
resultados = buscar(col, db, consulta)

# Preparar el contexto con los resultados recuperados
contexto = "\n".join(
    f"Imagen: {id_}, etiqueta: {diag}, distancia: {dist}"
    for id_, diag, dist in resultados
)

# Generar una respuesta usando ese contexto
prompt = f"""
Eres un asistente educativo para un proyecto de clasificación
de radiografías de tórax.

Utiliza únicamente la información del contexto.
No hagas diagnósticos médicos.
Si la información no es suficiente, indícalo.

Contexto recuperado:
{contexto}

Pregunta:
¿Qué información muestran los resultados recuperados?
"""

import time
from google.genai.errors import ServerError

respuesta = None

for intento in range(3):
    try:
        respuesta = cliente.models.generate_content(
            model="gemini-3.8-flash",
            contents=prompt
        )
        break
    except ServerError as e:
        if intento == 2:
            raise
        print(f"Gemini ocupado. Reintentando en 10 segundos...")
        time.sleep(10)

print(respuesta.text)


Basándome únicamente en el contexto proporcionado, los resultados recuperados muestran la siguiente información sobre 5 imágenes:

1. **Identificadores de las imágenes:**
   * NORMAL_IM-0115-0001
   * NORMAL_IM-0227-0001
   * NORMAL_IM-0278-0001
   * NORMAL_IM-0269-0001
   * NORMAL_IM-0135-0001

2. **Etiquetas asignadas:**
   * Todas las imágenes recuperadas tienen la etiqueta **NORMAL**.

3. **Valores de distancia:**
   * La imagen `NORMAL_IM-0115-0001` tiene una distancia de **0.0**.
   * Las cuatro imágenes restantes (`NORMAL_IM-0227-0001`, `NORMAL_IM-0278-0001`, `NORMAL_IM-0269-0001` y `NORMAL_IM-0135-0001`) tienen una distancia de **0.006**.


In [17]:

from google.colab import files

# 1. Subir una radiografía
archivos = files.upload()
ruta_consulta = next(iter(archivos))

# 2. Buscar las 5 imágenes más similares
print("Buscando imágenes similares...")
resultados = buscar(col, db, ruta_consulta, k=5)

# 3. Mostrar los resultados
contexto = "\n".join(
    f"Imagen: {id_} | Etiqueta: {diag} | Distancia: {dist}"
    for id_, diag, dist in resultados
)

print("\nResultados recuperados:")
print(contexto)

# 4. Preguntar al usuario
pregunta = input("\n¿Qué quieres saber sobre los resultados? ")

# 5. Consultar Gemini
prompt_corto = f"""
Analiza estos resultados de una búsqueda de radiografías similares.
Explica qué significan las etiquetas encontradas.
No confirmes un diagnóstico médico.

Resultados:
{contexto}

Pregunta: {pregunta}
"""

try:
    respuesta = cliente.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt_corto
    )
    print(respuesta.text)
except Exception as e:
    print(type(e).__name__, e)


Saving Xray sano.jpeg to Xray sano (1).jpeg
Buscando imágenes similares...

Resultados recuperados:
Imagen: PNEUMONIA_person101_virus_188 | Etiqueta: PNEUMONIA | Distancia: 0.09
Imagen: NORMAL_IM-0289-0001 | Etiqueta: NORMAL | Distancia: 0.092
Imagen: NORMAL_IM-0183-0001 | Etiqueta: NORMAL | Distancia: 0.093
Imagen: PNEUMONIA_person1035_bacteria_2969 | Etiqueta: PNEUMONIA | Distancia: 0.099
Imagen: PNEUMONIA_person103_virus_190 | Etiqueta: PNEUMONIA | Distancia: 0.099

¿Qué quieres saber sobre los resultados? Diagnostico


KeyboardInterrupt: 

In [18]:
prompt_corto = f"""
Analiza estos resultados de una búsqueda de radiografías similares.
Explica qué significan las etiquetas encontradas.
No confirmes un diagnóstico médico.

Resultados:
{contexto}

Pregunta: {pregunta}
"""

try:
    respuesta = cliente.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt_corto
    )
    print(respuesta.text)
except Exception as e:
    print(type(e).__name__, e)


A continuación se presenta el análisis de los resultados y el significado de las etiquetas obtenidas:

---

### 1. Significado de las etiquetas encontradas

* **NORMAL:** Indica radiografías de tórax de referencia que no presentan signos radiológicos evidentes de patología pulmonar activa ni consolidaciones infecciosas.
* **PNEUMONIA (Neumonía):** Indica radiografías donde se observan patrones compatibles con una infección pulmonar (como opacidades, infiltrados o consolidaciones). En los nombres de archivo se diferencian dos orígenes:
  * *virus:* Neumonía de origen viral (suele asociarse a patrones intersticiales difusos).
  * *bacteria:* Neumonía de origen bacteriano (frecuentemente asociada a consolidaciones lobares o focales).
* **Distancia:** Es una métrica matemática de similitud visual entre las características de la imagen de consulta y las de la base de datos. Cuanto menor es el valor, mayor es la semejanza visual en el espacio de características analizado por el algoritmo.

-

In [19]:

pregunta = "¿Cuáles son las limitaciones de utilizar estas cinco imágenes similares para interpretar la radiografía consultada?"

prompt = f"""
Responde en español, de forma clara y breve.
Utiliza los resultados recuperados como referencia.
No emitas diagnósticos médicos.

RESULTADOS:
{contexto}

PREGUNTA:
{pregunta}
"""

respuesta = cliente.models.generate_content(
    model="gemini-3.8-flash",
    contents=prompt
)

print(respuesta.text)


Las principales limitaciones de utilizar estos resultados son:

1. **Inconsistencia en las etiquetas:** Los resultados recuperados muestran clases contradictorias (3 casos de **PNEUMONIA** —con causas tanto virales como bacterianas— y 2 casos **NORMAL**), lo que genera ambigüedad.
2. **Distancias casi indistinguibles:** Las distancias de similitud son extremadamente cercanas entre sí (varían únicamente entre 0.090 y 0.099), lo que demuestra que el modelo no encuentra una separación clara entre las imágenes normales y patológicas.
3. **Ausencia de contexto clínico:** La similitud matemática visual no considera el historial, los síntomas ni la valoración de un profesional de la salud, por lo que estos resultados no son concluyentes ni sustituyen una evaluación médica.


In [20]:

print("Número de imágenes recuperadas:", len(resultados))
print("\nDetalle de los resultados:")

for i, (id_, diag, dist) in enumerate(resultados, start=1):
    print(f"{i}. Imagen: {id_} | Etiqueta: {diag} | Distancia: {dist}")


Número de imágenes recuperadas: 5

Detalle de los resultados:
1. Imagen: PNEUMONIA_person101_virus_188 | Etiqueta: PNEUMONIA | Distancia: 0.09
2. Imagen: NORMAL_IM-0289-0001 | Etiqueta: NORMAL | Distancia: 0.092
3. Imagen: NORMAL_IM-0183-0001 | Etiqueta: NORMAL | Distancia: 0.093
4. Imagen: PNEUMONIA_person1035_bacteria_2969 | Etiqueta: PNEUMONIA | Distancia: 0.099
5. Imagen: PNEUMONIA_person103_virus_190 | Etiqueta: PNEUMONIA | Distancia: 0.099


In [21]:

from collections import Counter

# Contar las etiquetas recuperadas
conteo = Counter(diag for _, diag, _ in resultados)

print("RESUMEN DE LA BÚSQUEDA")
print("----------------------")
print(f"Imágenes recuperadas: {len(resultados)}")

for etiqueta, cantidad in conteo.items():
    porcentaje = cantidad / len(resultados) * 100
    print(f"{etiqueta}: {cantidad} ({porcentaje:.1f} %)")

print("\nImagen más similar:")
id_, diag, dist = resultados[0]
print(f"Imagen: {id_}")
print(f"Etiqueta: {diag}")
print(f"Distancia: {dist}")


RESUMEN DE LA BÚSQUEDA
----------------------
Imágenes recuperadas: 5
PNEUMONIA: 3 (60.0 %)
NORMAL: 2 (40.0 %)

Imagen más similar:
Imagen: PNEUMONIA_person101_virus_188
Etiqueta: PNEUMONIA
Distancia: 0.09


In [37]:
import inspect
print(inspect.getsource(config))

MODELO = "gemini-embedding-2"   # si falla: "gemini-embedding-2-preview"
DIM = 768
BASE = "/content/drive/MyDrive/radiologia"
COLECCION = "rx_gemini"



In [23]:

!pip install -q streamlit


In [24]:

%%writefile app.py
import streamlit as st

st.set_page_config(
    page_title="Chatbot de radiografías",
    page_icon="🩻",
    layout="wide"
)

st.title("Chatbot educativo de radiografías de tórax")
st.write(
    "Busca radiografías similares y consulta sus etiquetas mediante Gemini. "
    "No sustituye una evaluación médica."
)

st.info(
    "Esta interfaz necesita conectarse a la base vectorial y a Gemini "
    "mediante las funciones y credenciales configuradas en el proyecto."
)

archivo = st.file_uploader(
    "Sube una radiografía de tórax",
    type=["png", "jpg", "jpeg"]
)

pregunta = st.text_input(
    "¿Qué quieres saber sobre los resultados?"
)

if st.button("Buscar y consultar"):
    if archivo is None:
        st.warning("Primero debes subir una radiografía.")
    elif not pregunta.strip():
        st.warning("Escribe una pregunta.")
    else:
        st.warning(
            "Falta conectar aquí la función buscar(), la base vectorial "
            "y la llamada a Gemini para ejecutar el flujo completo."
        )


Writing app.py


In [28]:

from google.colab import files
files.download("app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:

%%writefile requirements.txt
streamlit
tensorflow
numpy
pillow
google-genai
qdrant-client

Overwriting requirements.txt


In [30]:

from google.colab import files
files.download("requirements.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>